In [8]:
import torch
from torchvision import models, transforms
from PIL import Image
import cv2
import numpy as np
import urllib.request
import time
import os
import base64, json, urllib.request as ur, http.client

In [9]:
if not os.path.exists("aquarium.jpg"):
    raise FileNotFoundError(
        "\nPlease make sure you have the required input in this directory."
    )

if not os.path.exists("imagenet_classes.txt"):
    print("Downloading class labels...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
        "imagenet_classes.txt"
    )

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]


In [10]:
print("Loading ResNet18 model...")
try:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    print("  [OK] Pretrained weights loaded.")
    PRETRAINED = True
except Exception as e:
    print(f"  [INFO] Could not download pretrained weights ({e}). Using random init.")
    model = models.resnet18(weights=None)
    PRETRAINED = False

model.eval()

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    input_tensor = transform(img).unsqueeze(0)

    # TODO 4: Timing logic — measure inference latency in milliseconds
    start_time = time.perf_counter()          # Start timer
    with torch.no_grad():
        output = model(input_tensor)
    end_time = time.perf_counter()            # End timer

    latency_ms = (end_time - start_time) * 1000.0   # Convert seconds → ms

    probs = torch.nn.functional.softmax(output[0], dim=0)
    confidence, predicted_idx = torch.max(probs, 0)

    return categories[predicted_idx.item()], confidence.item(), latency_ms



Loading ResNet18 model...
  [OK] Pretrained weights loaded.


In [11]:
def simulate_turbidity(img_array):
    """
    Simulate murky water (blur/haze).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    # TODO 1: Gaussian blur simulates light scattering from suspended particles.
    # A pale blue-grey haze overlay mimics the reduced visibility in turbid water.
    blurred = cv2.GaussianBlur(img_array, (21, 21), sigmaX=0)
    haze = np.full_like(img_array, (210, 200, 180), dtype=np.uint8)  # BGR pale haze
    turbid = cv2.addWeighted(blurred, 0.55, haze, 0.45, 0)
    return turbid


In [12]:
def simulate_color_shift(img_array):
    """
    Simulate depth color loss (attenuate the red channel).
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    # TODO 2: OpenCV uses BGR — channel index 2 is Red.
    # Water absorbs red wavelengths rapidly with depth; we scale it to ~15%.
    shifted = img_array.astype(np.float32).copy()
    shifted[:, :, 2] *= 0.15       # Red channel (index 2 in BGR)
    shifted = np.clip(shifted, 0, 255).astype(np.uint8)
    return shifted

In [13]:
def simulate_sensor_noise(img_array):
    """
    Simulate low-light digital camera noise.
    Input: OpenCV image array (BGR)
    Output: Modified OpenCV image array
    """
    # TODO 3: Gaussian noise (mean=0, std=35) injected pixel-wise to mimic
    # thermal/read noise in low-light / high-ISO camera sensors.
    noise = np.random.normal(loc=0, scale=35, size=img_array.shape)
    noisy = img_array.astype(np.float32) + noise
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    return noisy


In [14]:
if __name__ == "__main__":

    all_images = [
        "aquarium.jpg",
        "jellyfish.jpg",
        "set_f20_SESR.png",
        "set_f46_SESR.png",
        "set_o20_SESR.png",
        "set_u106_SESR.png",
        "set_u113_SESR.png",
    ]

    results = []

    for base_image in all_images:
        if not os.path.exists(base_image):
            print(f"Skipping {base_image} — not found.")
            continue

        img = cv2.imread(base_image)
        stem = os.path.splitext(base_image)[0]

        turbid_path     = f"{stem}_turbid.jpg"
        colorshift_path = f"{stem}_colorshift.jpg"
        noise_path      = f"{stem}_noise.jpg"

        cv2.imwrite(turbid_path,     simulate_turbidity(img.copy()))
        cv2.imwrite(colorshift_path, simulate_color_shift(img.copy()))
        cv2.imwrite(noise_path,      simulate_sensor_noise(img.copy()))

        images_to_test = [
            ("Baseline (Clean)", base_image),
            ("Turbidity",        turbid_path),
            ("Color Shift",      colorshift_path),
            ("Sensor Noise",     noise_path),
        ]

        print(f"\n{'='*50}")
        print(f"IMAGE: {base_image}")
        print(f"{'='*50}")
        print("RUNNING VISION DIAGNOSTICS")

        for condition_name, file_path in images_to_test:
            label, conf, latency = predict(file_path)
            print(f"Condition : {condition_name}")
            print(f"Prediction: {label}")
            print(f"Confidence: {conf:.4f}")
            print(f"Latency   : {latency:.2f} ms")
            print("-" * 30)
            results.append({
                "image": base_image,
                "condition": condition_name,
                "label": label,
                "confidence": conf,
                "latency_ms": latency,
            })

    # ── Summary ───────────────────────────────────────────────────────────────
    print("\n\n" + "="*70)
    print("SUMMARY TABLE")
    print("="*70)
    print(f"{'Image':<25} {'Condition':<20} {'Confidence':>10} {'Latency(ms)':>12}")
    print("-"*70)
    for r in results:
        print(f"{r['image']:<25} {r['condition']:<20} {r['confidence']:>10.4f} {r['latency_ms']:>12.2f}")

    latencies = [r["latency_ms"] for r in results]
    avg_lat   = sum(latencies) / len(latencies)
    max_lat   = max(latencies)
    fps_budget = 1000.0 / 30.0   # 33.33 ms
    print(f"\nAvg inference latency : {avg_lat:.2f} ms")
    print(f"Max inference latency : {max_lat:.2f} ms")
    print(f"30 FPS budget (33.33 ms): {'WITHIN budget ✓' if max_lat < fps_budget else 'EXCEEDS budget ✗ — frames will drop'}")
    print(f"\nNote: pretrained={'YES' if PRETRAINED else 'NO (random init — network restricted)'}")



IMAGE: aquarium.jpg
RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: eel
Confidence: 0.4804
Latency   : 21.05 ms
------------------------------
Condition : Turbidity
Prediction: jacamar
Confidence: 0.0355
Latency   : 20.33 ms
------------------------------
Condition : Color Shift
Prediction: eel
Confidence: 0.5497
Latency   : 22.62 ms
------------------------------
Condition : Sensor Noise
Prediction: boa constrictor
Confidence: 0.1315
Latency   : 23.40 ms
------------------------------

IMAGE: jellyfish.jpg
RUNNING VISION DIAGNOSTICS
Condition : Baseline (Clean)
Prediction: jellyfish
Confidence: 0.9949
Latency   : 22.70 ms
------------------------------
Condition : Turbidity
Prediction: rapeseed
Confidence: 0.0616
Latency   : 22.04 ms
------------------------------
Condition : Color Shift
Prediction: jellyfish
Confidence: 0.9939
Latency   : 21.51 ms
------------------------------
Condition : Sensor Noise
Prediction: wool
Confidence: 0.3023
Latency   : 20.32 ms
----